In [ ]:
# We chose Ahya as an example for cell type annotation

import scanpy as sc

adata = sc.read_h5ad("/absolute/path/pairwise_h5ad/Ahya_lei_mc.h5ad")

print(adata)

print("obs:", adata.obs.head())       
print("var:", adata.var.head())       
print("obsm:", adata.obsm.keys())     
print("layers:", adata.layers.keys()) 
print("uns:", adata.uns.keys())       

In [ ]:
adata.obs["leiden"].unique()

In [ ]:
adata.var_names = adata.var_names.str.replace('-', '_', regex=False)
import anndata  
adata.raw = anndata.AnnData(adata.X.copy(), obs=adata.obs.copy(), var=adata.var.copy())


In [ ]:
print(adata.obs.columns)

adata.obs = adata.obs.rename(columns={"leiden": "cell_type"})
adata.__dict__['_raw'].__dict__['_var'] = adata.__dict__['_raw'].__dict__['_var'].rename(columns={'_index': 'features'})


print(adata.obs.columns)

adata.write("Ahya_leiden_metacell_data.h5ad")

In [ ]:
from samap.mapping import SAMAP
from samap.analysis import (get_mapping_scores, GenePairFinder,
                            sankey_plot, chord_plot, CellTypeTriangles, 
                            ParalogSubstitutions, FunctionalEnrichment,
                            convert_eggnog_to_homologs, GeneTriangles)
from samalg import SAM
import pandas as pd

## Loading in raw data

In [ ]:
fn1 = './Ahya_leiden_metacell_data.h5ad'
fn2 = '/absolute/path/pairwise_h5ad/Spis_25_nature.h5ad'

In [ ]:
filenames = {'ah':fn1,'sp':fn2}

In [ ]:
sm = SAMAP(
        filenames,
        f_maps = '/absolute/path/samap_maps/',
        save_processed=True ,
        keys = {'ah':'cell_type','sp':'cell_type'},
    )
sm.run(neigh_from_keys = {'ah':True,'sp':True})

In [ ]:
sm.sams['ah'].adata.obs['cell_type'].head()

In [ ]:
sm.sams['sp'].adata.obs['cell_type'].head()

In [ ]:
sm.run(pairwise=True)
samap = sm.samap # SAM object with three species stitched together

In [ ]:
sm.scatter()

In [ ]:
import pickle
with open('./sm_ahya_spis.pkl', 'wb') as file:
    pickle.dump(sm, file)
print("对象已保存到本地文件。")

Post-SAMap analysis

In [ ]:
import pickle


with open('./sm_ahya_spis.pkl', 'rb') as file:
    sm = pickle.load(file)

print("对象已成功加载。")


In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import networkx as nx
import umap
from scipy.sparse import csr_matrix
import seaborn as sns
import holoviews as hv

整合UMAP可视化

In [ ]:
samap = sm.samap
adata = samap.adata
adata

In [ ]:
sc.pl.umap(adata, color=['species'], title='UMAP projection colored by species', size=10)

## Calculating cell type mapping scores

sankey_plot

In [ ]:
keys = {'ah':'cell_type','sp':'cell_type'}
D,MappingTable = get_mapping_scores(sm,keys,n_top = 0)
D.head()
MappingTable.head()

sankey_plot(MappingTable, align_thr=0.05, species_order=['ah', 'sp'])


In [ ]:
import holoviews as hv
hv.extension('bokeh')

sankey_obj = sankey_plot(MappingTable, align_thr=0.15, species_order=['ah', 'sp'])
sankey_obj
sankey_obj.opts(frame_width=2000, frame_height=1600)
hv.save(sankey_obj, "./sankey_plot.html", resources='inline')

In [ ]:
MappingTable.index

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

gf_rows = MappingTable.index[MappingTable.index.str.startswith('ah_')]
sp_cols = MappingTable.columns[MappingTable.columns.str.startswith('sp_')]

def sort_gf_index(gf_list):
    def extract_number(x):
        try:
            return int(x.split('_')[1])
        except (IndexError, ValueError):
            return 0

    return sorted(gf_list, key=extract_number)

gf_rows_sorted = sort_gf_index(gf_rows.tolist())

FilteredMappingTable = MappingTable.loc[gf_rows_sorted, sp_cols]

print(FilteredMappingTable.head())

sns.set(style="white")

plt.figure(figsize=(20, 15))

sns.heatmap(
    FilteredMappingTable,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    linewidths=0.5,
    cbar_kws={"label": "Similarity Score"},
    annot_kws={"size": 8},
)

plt.title("Similarity Heatmap between Clusters", fontsize=18)
plt.xlabel("Clusters", fontsize=16)
plt.ylabel("Clusters", fontsize=16)

plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

plt.savefig("./ah_vs_sp_heatmap.pdf", format="pdf", bbox_inches="tight")

plt.show()

In [ ]:
def export_mapping_table_for_sankey(mapping_table, output_file='mapping_table.csv'):

    if not isinstance(mapping_table, pd.DataFrame):
        raise ValueError("mapping_table must be a pandas DataFrame")
    

    mapping_table.to_csv(output_file, index=True, header=True)
    print(f"Mapping table exported to {output_file}")
    print(f"Table shape: {mapping_table.shape}")
    print(f"Sample of the data:")
    print(mapping_table.iloc[:5, :5])
    
    return output_file


export_mapping_table_for_sankey(FilteredMappingTable, 'ahsp_samap_mapping_table.csv')